In [1]:
from pyprojroot import here
print("Project root:", here())

Project root: C:\Users\hanis\Flight\flight-delay-prediction


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load train data
df = pd.read_parquet(here("data/processed/train.parquet"))
print(df.columns)

Index(['Month', 'DayofMonth', 'DayOfWeek', 'FlightDate', 'Reporting_Airline',
       'DOT_ID_Reporting_Airline', 'IATA_CODE_Reporting_Airline',
       'Tail_Number', 'Flight_Number_Reporting_Airline', 'OriginAirportID',
       'OriginAirportSeqID', 'OriginCityMarketID', 'Origin', 'OriginCityName',
       'OriginState', 'OriginStateFips', 'OriginStateName', 'OriginWac',
       'DestAirportID', 'DestAirportSeqID', 'DestCityMarketID', 'Dest',
       'DestCityName', 'DestState', 'DestStateFips', 'DestStateName',
       'DestWac', 'CRSDepTime', 'DepDel15', 'DepTimeBlk', 'CRSArrTime',
       'ArrTimeBlk', 'CRSElapsedTime', 'Distance', 'DistanceGroup'],
      dtype='object')


## Drop Columns that correspond to the same thing

### Removing Redundant Features

Several columns in the dataset represent the same underlying information using different encodings (e.g., IDs, abbreviations, and descriptive names). Retaining all of them increases dimensionality without providing additional predictive value.

The following columns are removed:

- **Airline identifiers**: Keeping `Reporting_Airline` and dropping numerical and IATA code representations.
- **Origin location duplicates**: Removing airport market IDs, state codes, FIPS codes, and WAC codes while retaining the more interpretable city and state names.
- **Destination location duplicates**: Similar treatment applied to destination-related columns.
- **Airport sequence IDs**: These contain nearly the same information as airport IDs and therefore add little value.
- **Time block features**: `DepTimeBlk` and `ArrTimeBlk` are coarse categorical representations of scheduled departure and arrival times and are redundant when exact scheduled times are available.
- **Distance group**: A binned version of `Distance`; the continuous distance feature is retained instead.
- **Flight date**: Since the dataset already contains `Month`, `DayofMonth`, and `DayOfWeek`, the raw date column is removed.
- **City and State information**: `OriginCityName`, `OriginStateName`, `DestCityName`, and `DestStateName` were removed because they are higher-level geographic descriptions that can be uniquely inferred from the corresponding airport identifiers. Retaining both airport IDs and their associated city/state information would introduce redundant location information.

This preprocessing step reduces redundancy, simplifies the feature space, and improves model interpretability while preserving the information most relevant for flight delay prediction.

In [3]:
duplicates_to_drop = [
    # Airline duplicates
    'DOT_ID_Reporting_Airline',
    'IATA_CODE_Reporting_Airline',

    # Origin location duplicates
    'Origin',
    'OriginCityMarketID',
    'OriginState',
    'OriginStateFips',
    'OriginWac',

    # Destination location duplicates
    'Dest',
    'DestCityMarketID',
    'DestState',
    'DestStateFips',
    'DestWac',

    # Airport sequence IDs (almost same information)
    'OriginAirportSeqID',
    'DestAirportSeqID',

    # Dep and Arr times (every Blk is just a block of 1 hr after CRSTime)
    'ArrTimeBlk',
    'DepTimeBlk',

    # DistanceGroup is a binned version of Distance
    'DistanceGroup',

    # day and month are already present
    'FlightDate',

    # State and City are just Broader aspects of AirportID
    'OriginCityName','OriginStateName',
    'DestCityName', 'DestStateName'
]

df_clean = df.drop(columns= duplicates_to_drop)

## Drop columns with High Cardinality

#### Drop Tail_Number, Flight Number:
- They have a very high cardinality.
- we don't have lots of data per aircraft.
- Model may memorize specific planes rather than learn general patterns.


In [4]:
high_cardinality_cols = [
    'Tail_Number', 'Flight_Number_Reporting_Airline',
]

df_clean = df_clean.drop(columns=high_cardinality_cols)

#### Drop the month column as, our training set only contains 3 months
 A model trained only on Aug–Oct cannot learn from the Month column

In [5]:
df_clean = df_clean.drop(columns = ['Month'])

In [6]:
print(df_clean.columns)
print(df_clean.shape)

Index(['DayofMonth', 'DayOfWeek', 'Reporting_Airline', 'OriginAirportID',
       'DestAirportID', 'CRSDepTime', 'DepDel15', 'CRSArrTime',
       'CRSElapsedTime', 'Distance'],
      dtype='object')
(1791079, 10)


In [7]:
df_clean.to_parquet(here("data/processed/reduced.parquet"))